In [29]:
import sympy as sym
from sympy import *
import numpy as np
from tabulate import tabulate

In [30]:
#constants for 11B nuclei
Ispin = 3/2
w0 = 192.55 #Larmor Frequency for 11B (MHz)
whz = w0*10**6 # LArmor frequency in Hz

#factor q given in article
q = (3-4*Ispin*(Ispin + 1))/(16*(whz))

# # Coefficient for LHQ (cluster 1) from ASICS (in Hz)
# A_coeff = [-1.730215*10**3,-2.744504*10**3,-3.396561*10**3]
# B_coeff = [1.251296*10**3,2.561384*10**3,-1.133846*10**3]
# C_coeff = [3.573296*10**3,0.986618*10**3,-0.473004*10**3]
# D_coeff = [-0.060956*10**3,-0.376258*10**3,-0.980912*10**3]
# E_coeff = [-0.037878*10**3,-0.579924*10**3,-1.166911*10**3]

#Coefficient for LHQ (cluster 2) from ASICS (in Hz)
A_coeff = [-2.237*10**3,-2.631*10**3,-3.326*10**3]
B_coeff = [1.436*10**3,2.462*10**3,-1.178*10**3]
C_coeff = [-3.527*10**3,0.272*10**3,-0.705*10**3]
D_coeff = [0.223*10**3,-0.402*10**3,-0.926*10**3]
E_coeff = [0.005*10**3, -0.526*10**3,-1.129*10**3]



In [31]:
# Sort eigenvalues of tensors as per the convention defined in article
def sort_eigenvalues(Tensor):
    #Calculate Quadrupolar Tensor in PAS
    eigenvalues, eigenvectors = np.linalg.eig(Tensor)
    print(' Unsorted Eigenvalues:\n', eigenvalues, '\n')
    print(' Unsorted Eigenvectors:\n', eigenvectors, '\n')

    avg_tensor = np.mean(eigenvalues) # Tr(A)Quad/3
    eigenvalue_diff = eigenvalues - avg_tensor

    # Get the indices of the sorted eigenvalues based on the absolute values
    sorted_indices = np.argsort(np.abs(eigenvalue_diff))

    # Sort both eigenvalues and eigenvectors using the sorted indices
    sorted_eigenvalues = eigenvalues[sorted_indices]
    sorted_eigenvectors = eigenvectors[:, sorted_indices] # The normalized (unit “length”) eigenvectors, 
    #                                                       such that the column eigenvectors[:,i] is the eigenvector corresponding to the eigenvalue eigenvalues[i]
    
    print('Sorted Eigenvalues: \n', sorted_eigenvalues, '\n')
    print('Sorted Eigenvectors: \n', sorted_eigenvectors, '\n')
    return sorted_eigenvalues, sorted_eigenvectors,  avg_tensor, eigenvalues, eigenvectors

In [32]:
#Define symbol for quadrupolar tensor and force them to be real
AzzmAyy_Q,AzzmAxx_Q,AyymAxx_Q,Ayz_Q,Axz_Q,Axy_Q = sym.symbols('AzzmAyy_Q,AzzmAxx_Q,AyymAxx_Q,Ayz_Q,Axz_Q,Axy_Q', real=True)

#Variable for each equation set
quad_tensor = [(AzzmAyy_Q, Ayz_Q), (AzzmAxx_Q, Axz_Q), (AyymAxx_Q, Axy_Q)]

# List to hold solutions for quadrupolar tensor terms
solutions_Q = []

for i, (diagonal_diff, off_diagonal) in enumerate(quad_tensor):
    
    eq1 = sym.Eq(-((diagonal_diff)**2 - 4*(off_diagonal)**2)*((9*q/8)), D_coeff[i])
    eq2 = sym.Eq(off_diagonal*(diagonal_diff)*(9*q/2), E_coeff[i])

    # Solve the system
    solution = sym.solve([eq1, eq2], (diagonal_diff, off_diagonal))
    solutions_Q.append(solution)

# Assign the solutions to the respective variables
AzzmAyy_Q = [solutions_Q[0][0][0], solutions_Q[0][1][0]]
Ayz_Q = [solutions_Q[0][0][1], solutions_Q[0][1][1]]
AzzmAxx_Q = [solutions_Q[1][0][0], solutions_Q[1][1][0]]
Axz_Q = [solutions_Q[1][0][1], solutions_Q[1][1][1]]
AyymAxx_Q = [solutions_Q[2][0][0], solutions_Q[2][1][0]]
Axy_Q = [solutions_Q[2][0][1], solutions_Q[2][1][1]]

# print(solutions_Q)
# Print the final results for the variables
print(f"Azz - Ayy: {AzzmAyy_Q}, Ayz: {Ayz_Q}")
print(f"Azz - Axx: {AzzmAxx_Q}, Axz: {Axz_Q}")
print(f"Ayy - Axx: {AyymAxx_Q}, Axy: {Axy_Q}")  



Azz - Ayy: [-225602.852366100, 225602.852366100], Ayz: [1264.43108439228, -1264.43108439228]
Azz - Axx: [-172250.008466978, 172250.008466978], Axz: [-174219.289398915, 174219.289398915]
Ayy - Axx: [-246883.736120366, 246883.736120366], Axy: [-260898.274438530, 260898.274438530]


In [33]:
import itertools
# Generate all combinations of AzzmAxx, AyymAxx, and AzzmAyy
combinations = list(itertools.product(AzzmAxx_Q, AyymAxx_Q, AzzmAyy_Q))
print('Combination of (Azz - Axx), (Ayy - Axx), (Azz - Ayy): \n ', combinations)

Combination of (Azz - Axx), (Ayy - Axx), (Azz - Ayy): 
  [(-172250.008466978, -246883.736120366, -225602.852366100), (-172250.008466978, -246883.736120366, 225602.852366100), (-172250.008466978, 246883.736120366, -225602.852366100), (-172250.008466978, 246883.736120366, 225602.852366100), (172250.008466978, -246883.736120366, -225602.852366100), (172250.008466978, -246883.736120366, 225602.852366100), (172250.008466978, 246883.736120366, -225602.852366100), (172250.008466978, 246883.736120366, 225602.852366100)]


In [34]:

#Find Quadrupolar tensor diagonal elements

Axx1 = []; Axx2 = []; Axx3 = []
Ayy1 = []; Ayy2 = []; Ayy3 = []
Azz1 = []; Azz2 = []; Azz3 = []

# Initialize variables to track the best combination and minimum variation
best_combination = None
min_variation = float('inf')
for (AzzmAxx_val, AyymAxx_val, AzzmAyy_val) in combinations:
    # Solution 1
    Axx1_val = (-(AzzmAxx_val + AyymAxx_val)/3)
    Ayy1_val = Axx1_val + AyymAxx_val
    Azz1_val = Axx1_val + AzzmAxx_val

    #Save values
    Axx1.append(Axx1_val)
    Ayy1.append(Ayy1_val)
    Azz1.append(Azz1_val)

     # Solution 2
    Ayy2_val = -(AzzmAyy_val - AyymAxx_val) / 3
    Axx2_val = Ayy2_val - AyymAxx_val
    Azz2_val = Ayy2_val + AzzmAyy_val

     #Save values
    Axx2.append(Axx2_val)
    Ayy2.append(Ayy2_val)
    Azz2.append(Azz2_val)

    # Solution 3
    Azz3_val = (AzzmAxx_val + AzzmAyy_val) / 3
    Axx3_val = Azz3_val - AzzmAxx_val
    Ayy3_val = Azz3_val - AzzmAyy_val
    
    #Save values
    Axx3.append(Axx3_val)
    Ayy3.append(Ayy3_val)
    Azz3.append(Azz3_val)

    # Convert sympy Float to regular Python float for NumPy functions
    Axx1_val = float(Axx1_val)
    Axx2_val = float(Axx2_val)
    Axx3_val = float(Axx3_val)
    
    Ayy1_val = float(Ayy1_val)
    Ayy2_val = float(Ayy2_val)
    Ayy3_val = float(Ayy3_val)
    
    Azz1_val = float(Azz1_val)
    Azz2_val = float(Azz2_val)
    Azz3_val = float(Azz3_val)

    # Calculate variation (standard deviation) for Axx, Ayy, Azz
    variation_Axx = np.std([Axx1_val, Axx2_val, Axx3_val])
    variation_Ayy = np.std([Ayy1_val, Ayy2_val, Ayy3_val])
    variation_Azz = np.std([Azz1_val, Azz2_val, Azz3_val])

    total_variation = variation_Axx + variation_Ayy + variation_Azz

    # Update the best combination if the current one has less variation
    if total_variation < min_variation:
        min_variation = total_variation
        best_combination = (AzzmAxx_val, AyymAxx_val, AzzmAyy_val)
        best_Axx_Q = np.mean([Axx1_val, Axx2_val, Axx3_val])
        best_Ayy_Q = np.mean([Ayy1_val, Ayy2_val, Ayy3_val])
        best_Azz_Q = np.mean([Azz1_val, Azz2_val, Azz3_val])

#Get index for off-diagonal elements        
index_AzzmAxx = AzzmAxx_Q.index(best_combination[0])
best_Axz_Q = Axz_Q[index_AzzmAxx]

index_AyymAxx = AyymAxx_Q.index(best_combination[1])
best_Axy_Q = Axy_Q[index_AyymAxx]

index_AzzmAyy = AzzmAyy_Q.index(best_combination[2])
best_Ayz_Q = Ayz_Q[index_AzzmAyy]

# Print results
print("Axx1:", Axx1)
print("Axx2:", Axx2)
print("Axx3:", Axx3)

print("Ayy1:", Ayy1)
print("Ayy2:", Ayy2)
print("Ayy3:", Ayy3)

print("Azz1:", Azz1)
print("Azz2:", Azz2)
print("Azz3:", Azz3)

print("Best combination with minimum standard deviation:")
print("AzzmAxx:", best_combination[0])
print("AyymAxx:", best_combination[1])
print("AzzmAyy:", best_combination[2])
print("Axz:", best_Axz_Q)
print("Axy:", best_Axy_Q)
print("Ayz:", best_Ayz_Q)

print("Average of Axx1, Axx2, Axx3 with minimum standard deviation:", best_Axx_Q)
print("Average of Ayy1, Ayy2, Ayy3 with minimum standard deviation:", best_Ayy_Q)
print("Average of Azz1, Azz2, Azz3 with minimum standard deviation:", best_Azz_Q)


Axx1: [139711.248195781, 139711.248195781, -24877.9092177958, -24877.9092177958, 24877.9092177958, 24877.9092177958, -139711.248195781, -139711.248195781]
Axx2: [239790.108202277, 89388.2066248769, -89388.2066248769, -239790.108202277, 239790.108202277, 89388.2066248769, -89388.2066248769, -239790.108202277]
Axx3: [39632.3881892853, 190034.289766686, 39632.3881892853, 190034.289766686, -190034.289766686, -39632.3881892853, -190034.289766686, -39632.3881892853]
Ayy1: [-107172.487924584, -107172.487924584, 222005.826902570, 222005.826902570, -222005.826902570, -222005.826902570, 107172.487924584, 107172.487924584]
Ayy2: [-7093.62791808838, -157495.529495489, 157495.529495489, 7093.62791808838, -7093.62791808838, -157495.529495489, 157495.529495489, 7093.62791808838]
Ayy3: [92985.2320884076, -207818.571066393, 92985.2320884076, -207818.571066393, 207818.571066393, -92985.2320884076, 207818.571066393, -92985.2320884076]
Azz1: [-32538.7602711969, -32538.7602711969, -197127.917684774, -19712

In [35]:
#Define symbol for CSA tensor and force them to be real
Azz_s, Axx_s, Ayy_s, Ayz_s, Axz_s, Axy_s = sym.symbols('Azz_s,Axx_s,Ayy_s,Ayz_s,Axz_s,Axy_s', real=True)

#Variables for each equation
cs_tensor = [(Ayy_s, Azz_s, Ayz_s), # for x -> Abb = Ayy; Agg = Azz; Abg = Ayz
             (Axx_s, Azz_s, Axz_s), # for y -> Abb = Axx; Agg = Azz; Abg = Axz
             (Axx_s, Ayy_s, Axy_s)] # for z -> Abb = Axx; Agg = Ayy; Abg = Axy

#Store variables in dictionary for access
A = {
    'xx': best_Axx_Q, 'yy': best_Ayy_Q, 'zz': best_Azz_Q,
    'yz': best_Ayz_Q, 'zy': best_Ayz_Q,
    'xz': best_Axz_Q, 'zx': best_Axz_Q,
    'xy': best_Axy_Q, 'yx': best_Axy_Q,
}

#Define rotation tuple (a, b, g, bg, m)
rotations = [
    ('x', 'y', 'z', 'yz', 1),   # a = x, b = y, g = z, m = 1
    ('y', 'x', 'z', 'xz', 1),  # a = y, b = x, g = z, m = 1
    ('z', 'x', 'y', 'xy', -1)  # a = z, b = x, g = y, m = -1
]

# List to hold solutions
solutions_cs = []

for i, (Abb_s, Agg_s, Abg_s) in enumerate(cs_tensor):
    a, b, g, bg, m = rotations[i]
    eq1 = sym.Eq(
        (8*A[a+a]*(A[b+b] + A[g+g] - A[a+a]) + 16*(A[a+b]**2 + A[a+g]**2) + 5*(A[b+b]**2 + A[g+g]**2) + 28*A[b+g]**2 - 18*A[b+b]*A[g+g])*(q/8) - 0.5*(Abb_s + Agg_s)*whz, A_coeff[i]
        )
    
    eq2 = sym.Eq(
        m*(2*A[a+a]*(A[b+b] - A[g+g]) - 12*(A[a+b]**2 - A[a+g]**2) - A[b+b]**2 + A[g+g]**2)*(q/2) - 0.5*m*(Agg_s - Abb_s)*whz, B_coeff[i]
        )
    
    eq3 = sym.Eq(
        -m*(-2*A[a+a]*A[b+g] + 12*A[a+b]*A[a+g] + A[b+g]*(A[b+b] + A[g+g]))*q - m*Abg_s*whz, C_coeff[i]
    )
    # Solve the system
    solution = sym.solve([eq1, eq2, eq3], (Abb_s, Agg_s, Abg_s))
    solutions_cs.append(solution)

print(solutions_cs)

#saving solutions
Axx_s = np.mean([solutions_cs[1][Axx_s], solutions_cs[2][Axx_s]])
Ayy_s = np.mean([solutions_cs[0][Ayy_s], solutions_cs[2][Ayy_s]])
Azz_s = np.mean([solutions_cs[0][Azz_s], solutions_cs[1][Azz_s]])

Ayz_s = solutions_cs[0][Ayz_s]
Axz_s = solutions_cs[1][Axz_s]
Axy_s = solutions_cs[2][Axy_s]

print('Axx_s, Ayy_s, Azz_s, Ayz_s, Axz_s, Axy_s: \n', Axx_s, Ayy_s, Azz_s, Ayz_s, Axz_s, Axy_s)


[{Ayy_s: 1.01174266069376e-5, Azz_s: 5.84332679787683e-6, Ayz_s: 2.93617849711986e-5}, {Axx_s: 1.35690259862438e-5, Azz_s: 5.68476979433476e-6, Axz_s: -2.99771573641902e-6}, {Axx_s: 1.22714402505190e-5, Ayy_s: 7.08247316915093e-6, Axy_s: -3.32633136267263e-6}]
Axx_s, Ayy_s, Azz_s, Ayz_s, Axz_s, Axy_s: 
 1.29202331183814e-5 8.59994988804426e-6 5.76404829610579e-6 2.93617849711986e-5 -2.99771573641902e-6 -3.32633136267263e-6


In [36]:
#Quadrupolar Tensor in Tenon Frame
Q_T = np.zeros((3,3))
Q_T[0,0] = best_Axx_Q; Q_T[0,1] = best_Axy_Q; Q_T[0,2] = best_Axz_Q;
Q_T[1,0] = best_Axy_Q; Q_T[1,1] = best_Ayy_Q; Q_T[1,2] = best_Ayz_Q;
Q_T[2,0] = best_Axz_Q; Q_T[2,1] = best_Ayz_Q; Q_T[2,2] = best_Azz_Q;

print('Quadrupolar tensor (tenon frame): \n', Q_T, '\n')

#CSA tensor in tenon frame
CS_T = np.zeros((3,3))
CS_T[0,0] = Axx_s; CS_T[0,1] = Axy_s; CS_T[0,2] = Axz_s;
CS_T[1,0] = Axy_s; CS_T[1,1] = Ayy_s; CS_T[1,2] = Ayz_s;
CS_T[2,0] = Axz_s; CS_T[2,1] = Ayz_s; CS_T[2,2] = Azz_s;

print('Chemical Shift tensor (tenon frame): \n', CS_T)

Quadrupolar tensor (tenon frame): 
 [[ 139711.24819578 -260898.27443853 -174219.28939892]
 [-260898.27443853 -157495.52949549   -1264.43108439]
 [-174219.28939892   -1264.43108439   17784.28129971]] 

Chemical Shift tensor (tenon frame): 
 [[ 1.29202331e-05 -3.32633136e-06 -2.99771574e-06]
 [-3.32633136e-06  8.59994989e-06  2.93617850e-05]
 [-2.99771574e-06  2.93617850e-05  5.76404830e-06]]


In [37]:
#following the Voseggard et al. paper for principal frame parameters JOURNAL OF MAGNETIC RESONANCE, Series A 122, 111 – 119 ( 1996 ) ARTICLE NO. 0186

#Calculate Quadrupolar Tensor in PAS

sorted_eigenvalues_Q, sorted_eigenvectors_Q, quad_avg, eigenvalues_Q, eigenvectors_Q = sort_eigenvalues(Q_T)
# print('Sorted Eigenvalues of Quadrupolar diagonal matrix (|Ayy - Tr(A)/3| <= |Axx - Tr(A)/3| <= |Azz - Tr(A)/3|):\n', sorted_eigenvalues_Q, '\n') 

Vyy = (sorted_eigenvalues_Q[0])*(2*Ispin*(2*Ispin - 1)) 
Vxx = (sorted_eigenvalues_Q[1])*(2*Ispin*(2*Ispin - 1))
Vzz = (sorted_eigenvalues_Q[2])*(2*Ispin*(2*Ispin - 1)) 

print('Quadupolar Tensor Components Vyy, Vxx, Vzz: \n', Vyy, Vxx, Vzz)

print('================================================================================================ \n')

#Calculate CSA Tensor in PAS

sorted_eigenvalues_csa, sorted_eigenvectors_csa, csa_avg, eigenvalues_csa, eigenvectors_csa = sort_eigenvalues(CS_T)
# print('Sorted Eigenvalues of CSA diagonal matrix (|Ayy - Tr(A)/3| <= |Axx - Tr(A)/3| <= |Azz - Tr(A)/3|):\n', sorted_eigenvalues_csa, '\n') 

csyy = -(sorted_eigenvalues_csa[0]) 
csxx = -(sorted_eigenvalues_csa[1])
cszz = -(sorted_eigenvalues_csa[2]) 

print('CSA Tensor Components δyy, δxx, δzz: \n', csyy, csxx, cszz)



 Unsorted Eigenvalues:
 [ 359530.8065863  -334022.4815703   -25508.32501601] 

 Unsorted Eigenvectors:
 [[-0.81341854  0.5387342   0.21935298]
 [ 0.40945126  0.79815453 -0.44192649]
 [ 0.41315849  0.26965684  0.86981909]] 

Sorted Eigenvalues: 
 [ -25508.32501601 -334022.4815703   359530.8065863 ] 

Sorted Eigenvectors: 
 [[ 0.21935298  0.5387342  -0.81341854]
 [-0.44192649  0.79815453  0.40945126]
 [ 0.86981909  0.26965684  0.41315849]] 

Quadupolar Tensor Components Vyy, Vxx, Vzz: 
 -153049.95009603925 -2004134.8894217808 2157184.839517819

 Unsorted Eigenvalues:
 [ 3.73965689e-05  1.21021113e-05 -2.22144489e-05] 

 Unsorted Eigenvectors:
 [[ 0.17989236 -0.9836798  -0.00357579]
 [-0.7124085  -0.12777474 -0.6900346 ]
 [-0.6783162  -0.12667938  0.72376755]] 

Sorted Eigenvalues: 
 [ 1.21021113e-05  3.73965689e-05 -2.22144489e-05] 

Sorted Eigenvectors: 
 [[-0.9836798   0.17989236 -0.00357579]
 [-0.12777474 -0.7124085  -0.6900346 ]
 [-0.12667938 -0.6783162   0.72376755]] 

CSA Tensor Co

In [38]:
#Quadrupolar tensor parameters
cq = Vzz/10**6
etaq = (Vyy - Vxx)/Vzz

#CSA tensor parameters
iso_cs = np.mean([cszz, csyy, csxx]) 
csa = cszz - iso_cs

etas = (csyy - csxx)/csa


table = [['cq (MHz)', cq], ['etaq', etaq ], ['iso_cs (ppm)', iso_cs*10**6], ['csa (ppm)', csa*10**6], ['etas', etas] ] #converting Hz to ppm (Should be multiplied by 10**6)
print(tabulate(table, headers=['Quantity', 'Fit Value']))

Quantity        Fit Value
------------  -----------
cq (MHz)         2.15718
etaq             0.858102
iso_cs (ppm)    -9.09474
csa (ppm)       31.3092
etas             0.807892


In [39]:
print(sorted_eigenvectors_Q)

b = np.degrees(np.arccos(sorted_eigenvectors_Q[2,2]))
a = np.degrees(np.arctan(sorted_eigenvectors_Q[2,1]/sorted_eigenvectors_Q[2,0]))
g = np.degrees(np.arctan(-sorted_eigenvectors_Q[1,2]/sorted_eigenvectors_Q[0,2]))
print("Calculated Euler angles (degrees):")
print('alpha:', a, 'beta:', b, 'gamma:', g,'\n')

[[ 0.21935298  0.5387342  -0.81341854]
 [-0.44192649  0.79815453  0.40945126]
 [ 0.86981909  0.26965684  0.41315849]]
Calculated Euler angles (degrees):
alpha: 17.22421292168036 beta: 65.5965988170269 gamma: 26.719355341211823 



In [40]:
print(sorted_eigenvectors_Q [:, :])

print(sorted_eigenvectors_Q[2,2])

15 + 180

[[ 0.21935298  0.5387342  -0.81341854]
 [-0.44192649  0.79815453  0.40945126]
 [ 0.86981909  0.26965684  0.41315849]]
0.41315848895564217


195

In [41]:
# Find Rotation Matrix and Rotation angles

# *************** Calculation for CSA ************************
# Quadrupolar Tensor in PAS
Q_PAS = np.zeros((3,3))
Q_PAS[0,0] = Vxx/(2*Ispin*(2*Ispin - 1));
Q_PAS[1,1] = Vyy/(2*Ispin*(2*Ispin - 1));
Q_PAS[2,2] = Vzz/(2*Ispin*(2*Ispin - 1));

#CSA tensor in PAS
CS_PAS = np.zeros((3,3))
CS_PAS[0,0] = -csxx; 
CS_PAS[1,1] = -csyy;
CS_PAS[2,2] = -cszz;
print('Calculation for CSA Tensor: \n')

# Find eigenvalues and eigenvectors of original matrix

eigenvalues, eigenvectors = np.linalg.eig(CS_PAS) 
print('Eigenvalues of CSA (PAS) tensor \n', eigenvalues, '\n')
print('Eigenvectors of CSA (PAS) tensor \n', eigenvectors, '\n')
# Calculate the eigenvalues of the rotated matrix A_rot

eigenvalues_rot, eigenvectors_rot = np.linalg.eig(CS_T)
print('Eigenvalues of CSA (Tenon) tensor \n', eigenvalues_rot, '\n')
print('Eigenvectors of CSA (Tenon) tensor \n', eigenvectors_rot, '\n')

b = np.degrees(np.arccos(CS_T[2,2]))
a = np.degrees(np.arctan(CS_T[2,1]/CS_T[2,0]))
g = np.degrees(np.arctan(-CS_T[1,2]/CS_T[0,2]))
print("Calculated Euler angles (degrees):")
print(a, b, g,'\n')






Calculation for CSA Tensor: 

Eigenvalues of CSA (PAS) tensor 
 [ 3.73965689e-05  1.21021113e-05 -2.22144489e-05] 

Eigenvectors of CSA (PAS) tensor 
 [[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]] 

Eigenvalues of CSA (Tenon) tensor 
 [ 3.73965689e-05  1.21021113e-05 -2.22144489e-05] 

Eigenvectors of CSA (Tenon) tensor 
 [[ 0.17989236 -0.9836798  -0.00357579]
 [-0.7124085  -0.12777474 -0.6900346 ]
 [-0.6783162  -0.12667938  0.72376755]] 

Calculated Euler angles (degrees):
-84.17053855460004 89.99966974435974 84.17053855460004 

